In [7]:
from astropy.io import fits
import numpy as np
import pandas as pd
from astropy.cosmology import FlatLambdaCDM
import astropy.units as u

In [8]:
#flussi sono in 10^-16*erg/(s*cm^2)
OIII_fluxes = []
OIII_err = []
HA_fluxes = []
HA_err = []
HB_fluxes = []
HB_err = []
for i in range(1, 8):
    df = pd.read_csv(f"C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/UGC5823/UGC5823_2/FORS2.2022-01-07T07_40_46.357/Spectra_extraction/OIII_ext/3Sigma/Spectrum3s_1_noreg_gas_exp.csv")
    df = df.set_index('Name')
    df_HA = pd.read_csv(f"C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/UGC5823/UGC5823_2/FORS2.2022-01-07T07_40_46.357/Spectra_extraction/3Sigma/Spectrum3s_{i}_noreg_gas_exp.csv")
    df_HA = df_HA.set_index('Name')
    HA_fluxes.append(df_HA.loc['Halpha', 'Flux'])
    HA_err.append(df_HA.loc['Halpha', 'Error_flux'])
    HB_fluxes.append(df.loc['Hbeta', 'Flux'])
    HB_err.append(df.loc['Hbeta', 'Error_flux'])
    OIII_fluxes.append(df.loc['[OIII]5007', 'Flux'])
    OIII_err.append(df.loc['[OIII]5007', 'Error_flux'])

#Convertiamo in erg/(s*cm^2)
HA_fluxes = np.array(HA_fluxes) * 1e-16
HB_fluxes = np.array(HB_fluxes) * 1e-16
OIII_fluxes = np.array(OIII_fluxes) * 1e-16

HA_err = np.array(HA_err) * 1e-16
HB_err = np.array(HB_err) * 1e-16
OIII_err = np.array(OIII_err) * 1e-16

In [9]:
def L_C_err(L_OIII, L_OIII_err, L_c_OIII, HA_flux, HA_err, HB_flux, HB_err):
    balmer_decr_err = np.sqrt((HA_err/HA_flux)**2+(HB_err/HB_flux)**2)*(HA_flux/HB_flux)
    base_err = balmer_decr_err/3
    exp_err = base_err*2.94*((HA_flux/HB_flux)/3)**1.94
    exp = ((HA_flux/HB_flux)/3)**2.94
    L_c_OIII_err = np.sqrt((L_OIII_err/L_OIII)**2+(exp_err/exp)**2)*L_c_OIII
    return L_c_OIII_err

def LX_err_func(L_c_OIII, L_c_OIII_err):
    #Normalizzo
    L_c_OIII_norm = L_c_OIII/1e42
    L_c_OIII_norm_err = L_c_OIII_err/1e42
    
    #Log e errore del log
    log = np.log10(L_c_OIII/1e42)
    err_log = (1/(L_c_OIII_norm*np.log(10)))*L_c_OIII_norm_err

    #errore di m*x nella retta
    err_mx = 1.02*np.abs(log)*np.sqrt((0.06/1.02)**2+(err_log/log)**2)

    #errore log(Lx_norm) = q+mx
    err_log_LX_norm = np.sqrt(0.1**2+(err_mx)**2)
    log_LX_norm = 1.1+1.02*np.log10(L_c_OIII/1e42)
    
    err_LX = (10**log_LX_norm*np.log(10)*err_log_LX_norm)*1e42
    return err_LX

def L_bol_function(Lx, sigma_Lx): #Da Duras 2020
    lh = Lx/3.8e33 #Lx/L_sun
    A = np.log10(lh) / 11.48
    k_x = 15.33 * (1 + A**16.20)
    
    # Derivata dk_x/dlh
    dk_dlh = 15.33 * 16.20 * (A**15.20) * (1/11.48) * (1/(lh * np.log(10)))
    
    # Derivata dlh/dLx
    dlh_dLx = 1 / (3.8e33)
    
    # Derivata totale
    dk_dLx = dk_dlh * dlh_dLx
    
    # Errore su kx
    sigma_kx = np.abs(dk_dLx) * sigma_Lx

    Lbol = Lx*k_x
    sigma_Lbol = np.sqrt((sigma_kx/k_x)**2+(sigma_Lx/Lx)**2)*Lbol
    return Lbol, sigma_Lbol

In [10]:
#Calcolo luminosità OIII
cosmo = FlatLambdaCDM(H0=70 * u.km / u.s / u.Mpc, Tcmb0=2.725 * u.K, Om0=0.3)

red = np.loadtxt("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/UGC5823/UGC5823_2/FORS2.2022-01-07T07_40_46.357/Spectra_extraction/redshift.txt") 
z = np.zeros(len(red))
for i in range(len(red)):
    z[i] = red[i][0] #l'errore sul redshift è tascurabile (chiedere conferma)

D_L = cosmo.luminosity_distance(z).value  # Mpc
D_L_cm = D_L * 3.086e24 #ora è in cm 

L_OIII = 4*np.pi*D_L_cm**2*OIII_fluxes*(100/70) #é in erg/s
L_OIII_err = 4*np.pi*D_L_cm**2*OIII_err*(100/70)

#Correzione L_OIII per Balmer decrement
L_c_OIII = L_OIII*((HA_fluxes/HB_fluxes)/3)**2.94
L_c_OIII_err = L_C_err(L_OIII, L_OIII_err, L_c_OIII, HA_fluxes, HA_err, HB_fluxes, HB_err)

#Conversione in L_x
log_LX_norm = 1.1+1.02*np.log10(L_c_OIII/1e42) 
LX = 10**log_LX_norm * 1e42
LX_err = LX_err_func(L_c_OIII, L_c_OIII_err)

#Calcolo Lbol
Lbol, Lbol_err = L_bol_function(LX, LX_err)

AGN = [0, 0, 0, 0, 0, 0, 0]
for i in range(len(Lbol)):
    if AGN[i] == 1:
        Lbol[i] = Lbol[i]
        Lbol_err[i] = Lbol_err[i]
    else:
        Lbol[i]= np.nan
        Lbol_err[i] = np.nan

In [11]:
# Verifica che i valori siano ragionevoli
print("L_OIII (erg/s):", L_OIII)
print("L_c_OIII (erg/s):", L_c_OIII)
print("L_c_OIII/1e42:", L_c_OIII/1e42)
print("LX (erg/s):", LX)
print("Errore relativo su LX:", LX_err/LX)
print("L_bol:", Lbol)
print("errore rel Lbol:", Lbol_err/Lbol)

L_OIII (erg/s): [2.59246684e+38 2.59528983e+38 2.60942845e+38 2.60659757e+38
 2.60659757e+38 2.60942845e+38 2.61509494e+38]
L_c_OIII (erg/s): [5.81730098e+38 5.55323336e+40 4.20303349e+39 9.28179776e+38
 5.19034035e+38 7.80707019e+39 5.72857393e+39]
L_c_OIII/1e42: [0.00058173 0.05553233 0.00420303 0.00092818 0.00051903 0.00780707
 0.00572857]
LX (erg/s): [6.30980539e+39 6.59837270e+41 4.74279232e+40 1.01621299e+40
 5.61693960e+39 8.91944528e+40 6.50440317e+40]
Errore relativo su LX: [0.58181524 0.41051855 0.49655542 0.56085102 0.58715403 0.47367152
 0.48707607]
L_bol: [nan nan nan nan nan nan nan]
errore rel Lbol: [nan nan nan nan nan nan nan]


In [12]:
#salvataggio 
d1 = {'L_c_OIII': L_c_OIII, 'sigma_{L_c_OIII}': L_c_OIII_err, 'LX': LX, 'sigma_{LX}': LX_err, 'Lbol': Lbol, 'sigma_{Lbol}': Lbol_err}
df1 = pd.DataFrame(data=d1)
df1 = df1.map(lambda x: f'{x:.3g}')
df1.to_csv("Luminosities.csv", index= False)

## The same after the correction

areas_OIII = pd.read_csv("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/FIRSTJ110949/FIRSTJ110949_2/Contaminations/Areas_OIII_3s-OIII.csv",
                        index_col = 0)

def true_flux(OIII_fluxes, area):        
    riga_1 = OIII_fluxes[0]*area['Peak 1'].iloc[0]/100
    err_1 = riga_1*0.1
    
    #riga_2 = spectrum2 * area['Peak 2'].iloc[1]/100
   # err_2 = e_2 * area['Peak 2'].iloc[1]/100
    
    riga_3 = OIII_fluxes[1] * area['Peak 3'].iloc[2]/100
    err_3 = riga_3*0.1
    
    riga_4 = OIII_fluxes[2]*area['Peak 4'].iloc[3]/100
    err_4 = riga_4*0.1
    
    riga_5 = OIII_fluxes[3] * area['Peak 5'].iloc[4]/100
    err_5 = riga_5*0.1

    riga_6 = OIII_fluxes[4] * area['Peak 6'].iloc[5]/100
    err_6 = riga_6*0.1
    OIII_true = [riga_1, riga_3, riga_4, riga_5, riga_6]
    sigma_OIII_true = [err_1, err_3, err_4, err_5, err_6]
    return OIII_true, sigma_OIII_true

OIII_true, sigma_OIII_true = true_flux(OIII_fluxes, areas_OIII)

L_OIII_true = 4*np.pi*D_L_cm**2*OIII_true #é in erg/s
L_OIII_err_true = 4*np.pi*D_L_cm**2*sigma_OIII_true

#Correzione L_OIII per Balmer decrement
L_c_OIII_true = L_OIII_true*((HA_fluxes/HB_fluxes)/3)**2.94
L_c_OIII_err_true = L_C_err(L_OIII_true, L_OIII_err_true, L_c_OIII_true, HA_fluxes, HA_err, HB_fluxes, HB_err)

#Conversione in L_x
log_LX_norm_true = 1.1+1.02*np.log10(L_c_OIII_true/1e42) 
LX_true = 10**log_LX_norm_true * 1e42
LX_err_true = LX_err_func(L_c_OIII_true, L_c_OIII_err_true)

#Calcolo Lbol
Lbol_true, Lbol_err_true = L_bol_function(LX_true, LX_err_true)

# Verifica che i valori siano ragionevoli
print("L_OIII (erg/s):", L_OIII_true)
print("L_c_OIII (erg/s):", L_c_OIII_true)
print("L_c_OIII/1e42:", L_c_OIII_true/1e42)
print("LX (erg/s):", LX_true)
print("Errore relativo su LX:", LX_err_true/LX_true)
print("L_bol:", Lbol_true)
print("errore rel Lbol:", Lbol_err_true/Lbol_true)

#salvataggio 
d1 = {'L_c_OIII': L_c_OIII_true, 'sigma_{L_c_OIII}': L_c_OIII_err_true, 'LX': LX_true, 'sigma_{LX}': LX_err_true, 'Lbol': Lbol_true, 'sigma_{Lbol}': Lbol_err_true}
df1 = pd.DataFrame(data=d1)
df1 = df1.map(lambda x: f'{x:.3g}')
df1.to_csv("Luminosities_true.csv", index= False)